---

<div style="border:1px solid #ddd; border-radius:12px; padding:20px; box-shadow:0px 4px 10px rgba(0,0,0,0.1);">

<h1 style="color:#4facfe;">📉 Chun Analysis : Telco Customer Chun</h1>

<p style="font-size:15px;">
EDA et insight generation sur <a href="https://www.kaggle.com/datasets/blastchar/telco-customer-churn">Telco Customer Churn Dataset</a>.
</p>

<hr>

<p>
👤 <b>Kodjo Jean DEGBEVI</b><br>
📘 <a href="../README.md">Readme</a>
</p>
<p style="font-style: italic;">Copyright (c) 2026 Kodjo Jean DEGBEVI.</p>

</div>

## 📖 Dictionnaire des variables

**Groupe 1 — Profil et Démographie**
- `customerID` : Identifiant unique
- `gender` : Genre (Male, Female)
- `SeniorCitizen` : Senior (1) ou non (0)
- `Partner` : En couple (Yes, No)
- `Dependents` : A des personnes à charge (Yes, No)

**Groupe 2 — Compte et Contrat**
- `tenure` : Ancienneté en mois
- `Contract` : Type de contrat (Month-to-month, One year, Two year)
- `PaperlessBilling` : Facturation électronique (Yes, No)
- `PaymentMethod` : Méthode de paiement (Bank transfer, Credit card, Electronic check, Mailed check)

**Groupe 3 — Services Souscrits (Engagement)**
- `PhoneService` : Service téléphonique (Yes, No)
- `MultipleLines` : Lignes multiples (Yes, No, No phone service)
- `InternetService` : Type de connexion (DSL, Fiber optic, No)
- `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` : Services additionnels

**Groupe 4 — Revenus & Cible**
- `MonthlyCharges` : Frais mensuels
- `TotalCharges` : Frais totaux cumulés sur la durée de vie client
- `Churn` : Cible - A résilié ce mois-ci (Yes, No)

In [ ]:
from pathlib import Path
import sys
CURRENT_DIR = Path().resolve()
ROOT_DIR = CURRENT_DIR.parent
DATA_DIR = ROOT_DIR / 'data'

sys.path.append(str(ROOT_DIR))

In [ ]:
#%load_ext autoreload
#%autoreload 2
import importlib
importlib.reload(importlib.import_module('src.clean'))

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
pio.templates.default = "plotly_white"

from src.clean import clean_data, create_tenure_cohorts
from src.metrics import calculate_global_churn_rate, calculate_churn_financial_impact

## Chargement et première obersvations

In [3]:
df_raw = pd.read_csv(DATA_DIR / 'raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df_raw.shape

(7043, 21)

In [4]:
display(df_raw.sample(5))

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
1873,0973-KYVNF,Female,0,Yes,Yes,72,Yes,No,DSL,Yes,...,Yes,No,Yes,No,Two year,Yes,Credit card (automatic),70.65,5011.15,No
548,4676-MQUEA,Male,1,Yes,No,50,Yes,Yes,Fiber optic,Yes,...,Yes,No,Yes,No,One year,Yes,Bank transfer (automatic),101.90,5265.5,No
5134,8010-EZLOU,Male,1,No,No,15,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,80.20,1217.25,Yes
1958,3737-XBQDD,Male,0,No,No,24,Yes,Yes,Fiber optic,No,...,Yes,No,No,No,Month-to-month,Yes,Bank transfer (automatic),84.85,2048.8,No
3432,8174-TBVCF,Female,0,Yes,No,70,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),94.80,6859.05,No


In [29]:
df_raw.describe(include='all')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,7590-VHVEG,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,20.2,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


In [5]:
display(df_raw.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

None

- **Aucune donnée manaquante apparente.**

- **`customerID` pris en feature**

In [32]:
int(df_raw.duplicated().sum())

0

**Aucun doublon strict trouvé.**

### Action de nettoyage
Le dictionnaire cible indique des clients ayant une ancienneté nulle (`tenure` = 0) car ils viennent de souscrire. La variable `TotalCharges` en est impactée car vide (espace). Je vais la convertir en float.

In [6]:
df_clean = clean_data(df_raw)
df_clean['TotalCharges'].dtype
print(f"Valeurs nulles pures : {df_clean.isnull().sum().sum()}")

Valeurs nulles pures : 0


In [7]:
display(df_clean[df_clean['tenure'] == 0][['tenure', 'MonthlyCharges', 'TotalCharges']].head())

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,0.0
753,0,20.25,0.0
936,0,80.85,0.0
1082,0,25.75,0.0
1340,0,56.05,0.0


## Q1. Quel est le taux de churn global et la perte de revenus associée ?

**Objectif :** Constater le problème. On va évaluer le MRR (Monthly Recurring Revenue) et le chiffre d'affaires cumulé que représentent les clients ayant "churné", afin de mettre l'enjeu financier en évidence.

In [13]:
churn_stats = calculate_global_churn_rate(df_clean)
financial_impact = calculate_churn_financial_impact(df_clean)

# Répartition du Churn
fig_pie = px.pie(
    names=['Restés (No)', 'Partis (Yes)'],
    values=[churn_stats['churn_counts']['No'], churn_stats['churn_counts']['Yes']],
    title="Proportion des clients ayant Churné",
    color_discrete_sequence=['#2ecc71', '#e74c3c'],
    hole=0.4
)
fig_pie.update_traces(textinfo='percent+label')
fig_pie.show()

# Impact MRR
fig_bar = go.Figure(data=[
    go.Bar(name='MRR Conservé', x=['MRR'], y=[financial_impact['retained_mrr']], marker_color='#2ecc71'),
    go.Bar(name='MRR Perdu', x=['MRR'], y=[financial_impact['lost_mrr']], marker_color='#e74c3c')
])

fig_bar.update_layout(
    title="Impact Financier Mensuel (MRR)",
    barmode='stack',
    yaxis_title="Revenus ($)"
)
fig_bar.show()

In [11]:
print(f"Total Clients : {churn_stats['total_customers']}")
print(f"Taux de Churn (%) : {churn_stats['churn_rate']:.2f}%")
print(f"MRR Perdu ($) : {financial_impact['lost_mrr']:,.2f}")
print(f"Total CA Perdu (LTV cumulée) ($) : {financial_impact['lost_total_revenue']:,.2f}")

Total Clients : 7043
Taux de Churn (%) : 26.54%
MRR Perdu ($) : 139,130.85
Total CA Perdu (LTV cumulée) ($) : 2,862,926.90


---

### 💡Conclusion Q1 

Le taux de Churn s'élève à **26.54%** de la base client, représentant une fuite financière directe de **139 130 \$ de MRR** (Revenu Mensuel Récurrent), et une perte de valeur à vie (CLV) évaluée à près de **2.86 Millions $**. <br>
>*L'impact business est donc majeur et nécessite une stratégie de rétention agressive.*

## Q2. Analyse de Survie & Cohortes d'ancienneté

**Objectif :** Comprendre à quel moment du cycle de vie client l'attrition est la plus forte, en segmentant l'ancienneté (`tenure`) par tranches annuelles. Cela permettra d'optimiser le timing de nos actions de rétention.

In [21]:
df_clean = create_tenure_cohorts(df_clean)

cohort_data = df_clean.groupby('tenure_group', observed=False).agg(
    Total_Clients=('Churn', 'count'),
    Churners=('Churn', lambda x: (x == 'Yes').sum())
).reset_index()

cohort_data['Churn_Rate (%)'] = (cohort_data['Churners'] / cohort_data['Total_Clients']) * 100

fig_cohort = make_subplots(specs=[[{"secondary_y": True}]])

fig_cohort.add_trace(
    go.Bar(
        x=cohort_data['tenure_group'], 
        y=cohort_data['Total_Clients'], 
        name="Total Clients", 
        marker_color='#3498db', 
        opacity=0.6
    ),
    secondary_y=False,
)

fig_cohort.add_trace(
    go.Scatter(
        x=cohort_data['tenure_group'], 
        y=cohort_data['Churn_Rate (%)'], 
        name="Taux de Churn (%)", 
        mode='lines+markers+text',
        text=cohort_data['Churn_Rate (%)'].apply(lambda x: f'{x:.1f}%'),
        textposition="top center",
        marker=dict(color='#e74c3c', size=12),
        line=dict(width=3)
    ),
    secondary_y=True,
)

fig_cohort.update_layout(
    title="Analyse de Survie : Volume de client et Taux d'Attrition par Cohorte",
    xaxis_title="Ancienneté (Cohortes)",
    hovermode="x unified"
)
# Cacher les grilles de l'axe Y secondaire pour la propreté du graphique
fig_cohort.update_yaxes(title_text="Volume Clients", secondary_y=False)
fig_cohort.update_yaxes(title_text="Taux de Churn (%)", showgrid=False, secondary_y=True)

fig_cohort.show()

---

### 💡 Conclusion Q2

L'attrition se produit majoritairement au début de la relation client : **47.4% des clients partent durant les 12 premiers mois**.<br> Le modèle met en évidence un effet d'ancrage fort : *plus le client survit à la première ou aux deux premières années (où la volatilité est explosive), plus il est probable qu'il reste fidèle (6.6% de churn seulement après 5 ans)*. 
> L'effort de fidélisation doit donc cibler l'amélioration de **l'onboarding et des 6 à 12 premiers mois**.

## Q3. Le type de contrat et de facturation influencent-ils la rétention ?

**Objectif :** Analyser comment la friction administrative et le niveau d'engagement contractuel affectent le churn. L'hypothèse métier est qu'un faible engagement (Mois par Mois) ou un moyen de paiement ponctuel ("Electronic check") génèrent plus de fuites.

In [24]:
def get_churn_rate_by_group(df, group_col):
    res = df.groupby(group_col, observed=False)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index(name='Churn_Rate')
    return res

contract_churn = get_churn_rate_by_group(df_clean, 'Contract')
payment_churn = get_churn_rate_by_group(df_clean, 'PaymentMethod')
payment_churn['PaymentMethod_Short'] = payment_churn['PaymentMethod'].replace({
    'Bank transfer (automatic)': 'Bank Transfer (Auto)',
    'Credit card (automatic)': 'Credit Card (Auto)'
})

In [25]:
fig_admin = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Taux de Churn (%) - Type de Contrat", "Taux de Churn (%) - Moyen de Paiement"),
    horizontal_spacing=0.1
)

fig_admin.add_trace(
    go.Bar(
        x=contract_churn['Contract'], 
        y=contract_churn['Churn_Rate'],
        text=contract_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color=['#e74c3c' if x == 'Month-to-month' else '#2ecc71' for x in contract_churn['Contract']]
    ),
    row=1, col=1
)

fig_admin.add_trace(
    go.Bar(
        x=payment_churn['PaymentMethod_Short'], 
        y=payment_churn['Churn_Rate'],
        text=payment_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color=['#e74c3c' if x == 'Electronic check' else '#95a5a6' for x in payment_churn['PaymentMethod']]
    ),
    row=1, col=2
)

fig_admin.update_layout(
    title="Frictions Administratives : L'engagement retient le client",
    showlegend=False,
    height=450
)
fig_admin.update_yaxes(title_text="Churn Rate (%)", row=1, col=1)

fig_admin.show()

---

### 💡 Conclusion Q3

Absolument. Le niveau d'engagement est un marqueur fort : un contrat **sans engagement (Mois par mois) présente un taux de fuite gigantesque (42.7%)**, contre presque rien pour un contrat de 2 ans (2.8%). <br>
D'autre part, le paiement manuel récurrent (et non automatisé) comme le **"Chèque électronique" génère de la friction et se hisse en tête des départs (45.2%)**. 
> Mettre en place un prélèvement automatique ou un blocage contractuel est un pare-feu naturel contre l'attrition.

## Q4. Quels services supplémentaires retiennent le mieux les utilisateurs ?

**Objectif :** Vérifier "l'effet d'enfermement" (Vendor Lock-in). <br>Je vais analyser si la souscription à de multiples services techniques (sauvegarde, sécurité, support, etc.) retient le client. <br>Je regarderai aussi l'impact du type d'infrastructure (Fibre vs DSL).

In [ ]:
services = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_clean['Total_Services'] = (df_clean[services] == 'Yes').sum(axis=1)

# Les clients sans Internet sont de base moins exposés au Churn
df_internet = df_clean[df_clean['InternetService'] != 'No']

service_churn = df_internet.groupby('Total_Services')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index(name='Churn_Rate')
internet_churn = get_churn_rate_by_group(df_clean, 'InternetService')

In [28]:
fig_services = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Churn (%) par Nombre de Services (Clients Internet)", "Churn (%) par Infrastructure"),
    horizontal_spacing=0.1
)

# Nombre de services add
fig_services.add_trace(
    go.Bar(
        x=service_churn['Total_Services'].astype(str) + " Svc", 
        y=service_churn['Churn_Rate'],
        text=service_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color='#3498db'
    ),
    row=1, col=1
)

# Type d'Internet
fig_services.add_trace(
    go.Bar(
        x=internet_churn['InternetService'], 
        y=internet_churn['Churn_Rate'],
        text=internet_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color=['#e74c3c' if x == 'Fiber optic' else '#95a5a6' for x in internet_churn['InternetService']]
    ),
    row=1, col=2
)

fig_services.update_layout(
    title="Dépendance Technologique & Infrastructure",
    showlegend=False,
    height=450
)
fig_services.update_yaxes(title_text="Churn Rate (%)", row=1, col=1)

fig_services.show()

**Plus un client souscrit à des options, moins il part.** <br> Un client ne possédant que l'internet "Nu" a 52.2% de risque de partir, contre seulement 5.3% s'il utilise 6 services de l'écosystème.<br> Par contre, un point d'attention critique est révélé : l'abonnement en **Fibre Optique subit un taux de désabonnement massif (41.9%)** comparativement à l'ADSL (19%). Regardons cela de plus prêt.

#### La Fibre Optique

In [47]:
fiber_df = df_clean[df_clean['InternetService'] == 'Fiber optic']
fiber_support_churn = get_churn_rate_by_group(fiber_df, 'TechSupport')

avg_price = df_clean.groupby('InternetService')['MonthlyCharges'].mean().reset_index()

fig_fiber = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Prix Mensuel Moyen par Offre ($)", "Fibre optique : Churn (%) selon le Support Technique"),
    horizontal_spacing=0.1
)

# Prix
fig_fiber.add_trace(
    go.Bar(
        x=avg_price['InternetService'], 
        y=avg_price['MonthlyCharges'],
        text=avg_price['MonthlyCharges'].apply(lambda x: f'${x:.0f}'),
        textposition='auto',
        marker_color=['#95a5a6', '#e74c3c', '#bdc3c7']
    ),
    row=1, col=1
)

# 2. TechSupport sur la Fibre
fig_fiber.add_trace(
    go.Bar(
        x=fiber_support_churn['TechSupport'], 
        y=fiber_support_churn['Churn_Rate'],
        text=fiber_support_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color=['#e74c3c' if x == 'No' else '#2ecc71' for x in fiber_support_churn['TechSupport']]
    ),
    row=1, col=2
)

fig_fiber.update_layout(
    title="Le paradoxe sur la Fibre Optique",
    showlegend=False, 
    height=450
)
fig_fiber.show()

Ce graphique illustre l'hémorragie sur la Fibre : **La fibre est le produit Premium (91\$/mois vs 58\$ pour l'ADSL)**. <br> Attentes hautes = risque élevé. <br>Si le client paie ce prix mais n'a **pas de Support Technique (TechSupport = No), son taux de désabonnement explose à 49.4%**. <br>Avec support, il chute à 22.6%.<br> 
> *Ma conclusion est donc que le problème est fortement lié à u**ne promesse qualité non tenue combinée à un défaut d'accompagnement en cas de panne***.

---

### 💡 Conclusion Q4

L'écosystème de services agit comme une puissante barrière de rétention (Vendor Lock-in) : **tous les services additionnels ajoutés au contrat diminuent le risque de départ**. 
<br>
Cependant, le **Support Technique (`TechSupport`)** se démarque comme le service le plus vital, car il colmate l'hémorragie massive observée sur l'offre Fibre Optique (qui est pourtant le produit le plus cher). 
> Il est impératif d'utiliser les services additionnels non seulement comme des sources de revenus supplémentaires (Cross-selling), mais surtout comme des **outils de fidélisation**, en envisageant par exemple d'inclure un support technique premium d'office dans les offres onéreuses.

## Q5. Quel profil démographique est le plus volatil ?

**Objectif :** Évaluer l'impact de la situation familiale (En couple, avec enfants) et de la tranche d'âge (Senior) sur la stabilité du client. Un profil isolé ou âgé présente-t-il un risque spécifique que nous pourrions cibler marketing-ment ?

In [41]:
df_clean['Family_Stability'] = (df_clean['Partner'] == 'Yes').astype(int) + (df_clean['Dependents'] == 'Yes').astype(int)

family_churn = get_churn_rate_by_group(df_clean, 'Family_Stability')
senior_churn = get_churn_rate_by_group(df_clean, 'SeniorCitizen')

In [42]:
fig_demo = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Churn (%) par Indice de Stabilité Familiale", "Churn (%) des Profils Seniors"),
    horizontal_spacing=0.1
)

fig_demo.add_trace(
    go.Bar(
        x=family_churn['Family_Stability'].astype(str) + " lien(s)", 
        y=family_churn['Churn_Rate'],
        text=family_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color=['#e74c3c' if x == 0 else ('#3498db' if x == 1 else '#2ecc71') for x in family_churn['Family_Stability']]
    ),
    row=1, col=1
)

fig_demo.add_trace(
    go.Bar(
        x=senior_churn['SeniorCitizen'], 
        y=senior_churn['Churn_Rate'],
        text=senior_churn['Churn_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='auto',
        marker_color=['#e74c3c' if x == 'Yes' else '#95a5a6' for x in senior_churn['SeniorCitizen']]
    ),
    row=1, col=2
)

fig_demo.update_layout(
    title="Segmentation Démographique & Risque d'Isolement",
    showlegend=False,
    height=450
)
fig_demo.update_yaxes(title_text="Churn Rate (%)", row=1, col=1)

fig_demo.show()

---
### 💡 Conclusion Q5 

Le "Client à risque" démographique est très clair : **il est isolé (Célibataire sans enfants = 34.2% de fuite)**, et s'il est **Senior, le risque est encore plus grand**. <br>
À l'inverse, l'ancrage familial (Couple avec enfants) agit comme une barrière au changement impressionnante, faisant chuter le churn à 14.2%. <br> Les coûts de sortie psychologiques (faire changer d'opérateur à toute la famille, interruption des services internet des enfants) sont plus élevés.

## Q6. Quelles actions spécifiques mettre en œuvre pour réduire le churn ? 

---

#### **Mes recommandations et plans d'actions**

Sur la base de mon diagnostic de l'attrition (~2.86M$ de CA potentiellement évaporé), voici les 4 piliers d'actions correctives à déployer en priorité par la direction :

**1. Verrouiller l'Engagement Contractuel et Financier (cf. Q3)**
- **Contrat :** Proposer des avantages (ex: mois offert, upgrade débit) pour inciter au passage d'un contrat `Month-to-month` vers un contrat `One year` ou `Two year`.
- **Paiement :** Inciter fortement au prélèvement automatique (ex: remise de 2$/mois sur la facture) pour éradiquer le paiement manuel par "Electronic check" responsable de la majorité des frictions (45.3% de churn).

**2. Opération "Sauvetage Fibre Optique" (cf. Q4)**
- L'offre Fibre à 91$/mois génère une insatisfaction massive en cas de problème. Il faut impérativement **inclure le `TechSupport` gratuitement** (ou le packager d'office en "Fibre Premium"). Un client Fibre optique sans assistance technique a presque 1 chance sur 2 de fuir (49.4%). S'il est accompagné, le risque chute de moitié (22.6%).

**3. Focus sur l'Onboarding des "Profils à Risque" (cf. Q2 & Q5)**
- **Timing :** 47% des départs ont lieu durant la première année. Il faut créer un programme de "Gants Blancs" (appels de courtoisie, aide à l'installation) durant les 6 premiers mois.
- **Ciblage :** Prioriser cette surveillance sur les profils les plus volatils : les **Profils isolés (Célibataires sans enfants)** et les **Seniors**.

**4. Multiplier le "Cross-Selling" d'Écosystème (cf. Q4)**
- Utiliser la puissance prouvée de l'enfermement technologique (Vendor Lock-in). Pousser la souscription de services de sécurité ou de backup à bas coût. Dès qu'un client franchit le cap des 3 ou 4 services intégrés à sa box, l'inertie s'installe et sa probabilité de départ s'effondre.